In [10]:
import pandas as pd
import json
from api_caller import call_api
from concurrent.futures import ThreadPoolExecutor, as_completed

In [11]:
# This method is used to combine parts of the speeches with the same ID.
def collapse(series):
    unique_vals = series.dropna().unique()
    if len(unique_vals) == 1:
        return unique_vals[0]
    else:
        return " ".join(map(str, series))


def prepare_dataset_pandas() -> pd.DataFrame:
    splits = {
        "train": "./data/train-00000-of-00001.parquet",
        #"dev":   "data/dev-00000-of-00001.parquet",
        #"test":  "data/test-00000-of-00001.parquet",
    }

    dfs = [
        pd.read_parquet(path)
        for path in splits.values()
    ]

    df = pd.concat(dfs, ignore_index=True)

    df_grouped = (
        df
        .groupby("speech", as_index=False)
        .agg(collapse)
    )

    df_grouped.drop(columns=["csv_index", "__index_level_0__", "split"], inplace=True)
    
    return df_grouped

In [12]:
#df = prepare_dataset_pandas()
#df.to_csv("data/train-00000-of-00001.csv")
df = pd.read_csv("data/train-00000-of-00001.csv")
df["EU Party"] = df["EU Party"].apply(lambda x: x.split('/')[-1])

df = df[~df["EU Party"].str.contains(r"\bNA\b", case=False, na=False)]

df_subset = df.iloc[2000:3000].copy()
#len(df)

In [13]:
def classify_batch(texts):
    joined_text = "\n\n".join(
        [f"ID {i}: {text}" for i, text in enumerate(texts)]
    )

    user_prompt = f"""You are a political ideology detection system.

Your task is to detect STRONG ideological value expression.

A speech should receive a high score ONLY IF it clearly expresses
a political principle or ideological position that could be mapped
onto a political survey (e.g., redistribution, EU integration,
national sovereignty, migration, social equality, market regulation,
democracy, rule of law, etc.).

STRICT CRITERIA:

The speech MUST:
- Advocate or oppose a political principle
- Express how society, the EU, or government SHOULD be structured
- Reveal a stable ideological commitment attributable to the speaker or their party

The speech must NOT be classified as value-expressing if it:
- Expresses generic hope or praise
- Uses polite or diplomatic language
- Evaluates events without ideological reasoning
- Contains general positive or negative sentiment only

IMPORTANT:
Generic approval is NOT ideological.

Be extremely conservative.
If the speech does not clearly state a political principle,
assign a score below 0.2.

Scoring guide:
0.0-0.2 → no ideological value content
0.3-0.5 → weak or vague value signals
0.6-0.8 → clear ideological positioning
0.9-1.0 → strong, explicit ideological commitment

Return STRICT JSON in the same order:
[
  {{
    "score": 0.0-1.0,
    "reason": "..."
  }}
]

IMPORTANT: Always return valid JSON array with exactly {len(texts)} entries, 
even if the score is 0.0 for some items.

Texts:
{joined_text}

"""

    response = call_api(user_prompt)

    return json.loads(response["choices"][0]["message"]["content"])

def process_batch(i, batch_size):
    print(i/len(df_subset)*100)
    batch_texts = df_subset["en"].iloc[i:i+batch_size].tolist()
    return classify_batch(batch_texts)

In [14]:

# Process in chunks for speed
batch_size = 10
indices = list(range(0, len(df_subset), batch_size))
results = []

'''
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(process_batch, i, batch_size): idx
        for idx, i in enumerate(indices)
    }

    ordered_batches = [None] * len(indices)

    for future in as_completed(futures):
        idx = futures[future]
        ordered_batches[idx] = future.result()

results = []
for batch in ordered_batches:
    results.extend(batch)
'''

'\nwith ThreadPoolExecutor(max_workers=4) as executor:\n    futures = {\n        executor.submit(process_batch, i, batch_size): idx\n        for idx, i in enumerate(indices)\n    }\n\n    ordered_batches = [None] * len(indices)\n\n    for future in as_completed(futures):\n        idx = futures[future]\n        ordered_batches[idx] = future.result()\n\nresults = []\nfor batch in ordered_batches:\n    results.extend(batch)\n'

In [15]:
for i in range(0, len(df_subset), batch_size):
    print(f"Progress: {i/len(df_subset)*100}%")
    batch_texts = df_subset["en"].iloc[i:i+batch_size].tolist()
    batch_results = classify_batch(batch_texts)
    results.extend(batch_results)

Progress: 0.0%
Progress: 1.0%
Progress: 2.0%
Progress: 3.0%
Progress: 4.0%
Progress: 5.0%
Progress: 6.0%
Progress: 7.000000000000001%
Progress: 8.0%
Progress: 9.0%
Progress: 10.0%
Progress: 11.0%
Progress: 12.0%
Progress: 13.0%
Progress: 14.000000000000002%
Progress: 15.0%
Progress: 16.0%
Progress: 17.0%
Progress: 18.0%
Progress: 19.0%
Progress: 20.0%
Progress: 21.0%
Progress: 22.0%
Progress: 23.0%
Progress: 24.0%
Progress: 25.0%
Progress: 26.0%
Progress: 27.0%
Progress: 28.000000000000004%
Progress: 28.999999999999996%
Progress: 30.0%
Progress: 31.0%
Progress: 32.0%
Progress: 33.0%
Progress: 34.0%
Progress: 35.0%
Progress: 36.0%
Progress: 37.0%
Progress: 38.0%
Progress: 39.0%
Progress: 40.0%
Progress: 41.0%
Progress: 42.0%
Progress: 43.0%
Progress: 44.0%
Progress: 45.0%
Progress: 46.0%
Progress: 47.0%
Progress: 48.0%
Progress: 49.0%
Progress: 50.0%
Progress: 51.0%
Progress: 52.0%
Progress: 53.0%
Progress: 54.0%
Progress: 55.00000000000001%
Progress: 56.00000000000001%
Progress: 56.999

In [ ]:
df_subset["reason"] = [r["reason"] for r in results]
df_subset["score"] = [r["score"] for r in results]
final_df = df_subset[["en", "score", "reason", "EU Party"]]
final_df.to_csv(
    "scored_data_gpt.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
#(final_df["score"] > 0.5).sum()
#len(df)
len(df_subset) 
len(results)

1000

In [ ]:
df_old = pd.read_csv("scored_data_gpt_first100.csv",sep=";",
    encoding="utf-8-sig")
combined_df = pd.concat([df_old, final_df], ignore_index=True)
(combined_df[combined_df["score"]>0.0]["EU Party"].value_counts()-combined_df[combined_df["score"]>0.5]["EU Party"].value_counts())/combined_df[combined_df["score"]>0.0]["EU Party"].value_counts()
combined_df.to_csv(
    "scored_data_gpt_first1000.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)